In [1]:
import cv2
import torch
import numpy as np
import onnxruntime as ort
import ipywidgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar
from jetcam.csi_camera import CSICamera
from torch2trt import TRTModule
from utils import preprocess

# 1. Hardware
car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

# 2. Road Following Model (TensorRT - Twoja mocna strona)
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth'))

# 3. YOLO Model (ONNX z akceleracją TensorRT)
providers = [
    ('TensorrtExecutionProvider', {'device_id': 0, 'trt_fp16_enable': True}), 
    'CUDAExecutionProvider'
]
yolo_session = ort.InferenceSession("yolov4_1_3_224_224_pacholek.onnx", providers=providers)
yolo_input_name = yolo_session.get_inputs()[0].name

# 4. UI
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

# Funkcja pomocnicza dla Road Following
def road_inference(frame):
    img = preprocess(frame).half() # Używamy Twojej funkcji preprocess i trybu FP16
    output = model_road(img).detach().cpu().numpy().flatten()
    return float(output[0])

print("Inicjalizacja zakończona ✔ (Modele Road i YOLO gotowe)")

Image(value=b'', format='jpeg', height='224', width='224')

Inicjalizacja zakończona ✔ (Modele Road i YOLO gotowe)


In [19]:
SPEED           = 0.36
K_LANE          = 2
AVOID_GAIN      = 0.8      
DANGER_Y        = 0.1      # 0.25 było zbyt wysoko na obrazie (za daleko)
AREA_THRESHOLD  = 0.018    # Obniżone, by łatwiej "zaskoczyło"
DEAD_ZONE       = 0       # Mniejszy dead_zone = szybsza reakcja
STEERING_BIAS   = 0.03
CONF_LIMIT      = 0.85      # 0.90 to za dużo dla Jetsona w ruchu
AVOID_HOLD_TIME = 0.5      # 1.5s to za długo, auto nie wróci na tor      # Podtrzymanie uniku w sekundach    


print(f"Parametry załadowane! Próg wielkości (AREA): {AREA_THRESHOLD}")

Parametry załadowane! Próg wielkości (AREA): 0.018


In [22]:
try:
    while True:
        frame = camera.value
        if frame is None: continue
        
        # 1. Obliczamy kierunek drogi (RFM)
        rfm_steer = road_inference(frame)
        # Lane center w skali 224px względem środka (112)
        lane_center_px = 112 + (rfm_steer * 112 * K_LANE)

        # 2. Detekcja przeszkód (YOLO)
        img_yolo = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).transpose((2, 0, 1)).astype(np.float32) / 255.0
        blob = np.expand_dims(img_yolo, axis=0)
        yolo_outs = yolo_session.run(None, {yolo_input_name: blob})
        
        boxes = np.squeeze(yolo_outs[0])
        scores = np.squeeze(yolo_outs[1])
        if len(scores.shape) > 1: scores = scores[:, 0]
        
        avoidance = 0
        best_conf = 0
        best_area = 0

        if len(scores) > 0:
            best_idx = np.argmax(scores)
            best_conf = scores[best_idx]
            
            if best_conf > CONF_LIMIT:
                box = boxes[best_idx]
                x1, y1, x2, y2 = (box * 224).astype(int)
                best_area = (box[2] - box[0]) * (box[3] - box[1])
                
                cx = (x1 + x2) / 2
                y_bottom = y2 / 224.0

                # Logika uniku względem wyliczonego środka toru (lane_center_px)
                if y_bottom > DANGER_Y:
                    relative_dist = cx - lane_center_px
                    
                    if abs(relative_dist) > DEAD_ZONE:
                        # Progresywny unik: im pachołek niżej, tym silniejszy skręt
                        weight = (y_bottom - DANGER_Y) / (1.0 - DANGER_Y)
                        avoidance = -AVOID_GAIN * weight if relative_dist > 0 else AVOID_GAIN * weight

                # Rysowanie boxa i info
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"C:{best_conf:.2f} A:{best_area:.3f}", (x1, y1-5), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        # 3. Wyjście na Hardware (Pamiętaj o swoim minusie!)
        final_steering = np.clip(rfm_steer + avoidance + STEERING_BIAS, -1.0, 1.0)
        car.steering = -final_steering 
        car.throttle = -SPEED

        # 4. Debug Video Overlay
        # Środek toru wg RFM (Zielona linia)
        cv2.line(frame, (int(lane_center_px), 0), (int(lane_center_px), 224), (0, 255, 0), 1)
        # Linia progu DANGER (Żółta)
        dy_px = int(DANGER_Y * 224)
        cv2.line(frame, (0, dy_px), (224, dy_px), (0, 255, 255), 2)
        
        image_widget.value = bgr8_to_jpeg(frame)
        
        print(f"\rST: {'AVOID' if avoidance != 0 else 'RFM  '} | Conf:{best_conf:.2f} | Steer:{-final_steering:.2f} | Avoid:{avoidance:.2f}", end="")

except KeyboardInterrupt:
    print("\nZatrzymano.")
finally:
    car.throttle = 0.0
    car.steering = STEERING_BIAS

ST: RFM   | Conf:0.00 | Steer:-0.08 | Avoid:0.006
Zatrzymano.
